In [ ]:
import requests
import pandas as pd
import os


def fetchHyperliquidLeaderboard(outputPath="../output/hyperliquidLeaderboard.parquet"):
    """
    Fetch Hyperliquid Mainnet leaderboard data and save to a Parquet file.
    """
    url = "https://stats-data.hyperliquid.xyz/Mainnet/leaderboard"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    rows = data.get("leaderboardRows", [])
    records = []

    for row in rows:
        record = {
            "ethAddress": row.get("ethAddress"),
            "accountValue": row.get("accountValue"),
            "prize": row.get("prize"),
            "displayName": row.get("displayName"),
        }

        # windowPerformances: list of [windowName, metricsDict]
        wp = row.get("windowPerformances", [])
        for entry in wp:
            if len(entry) == 2:
                winName, metrics = entry
                for metric in ["pnl", "roi", "vlm"]:
                    colName = f"{winName}_{metric}"
                    record[colName] = metrics.get(metric)

        records.append(record)

    df = pd.DataFrame(records)

    # --- Type conversions ---
    # Strings
    df["ethAddress"] = df["ethAddress"].astype(str)
    df["displayName"] = df["displayName"].where(df["displayName"].notna(), None)

    # Integer
    df["prize"] = df["prize"].astype("int64")

    # Numeric strings -> float64
    numericCols = [
        "accountValue",
        "day_pnl", "day_roi", "day_vlm",
        "week_pnl", "week_roi", "week_vlm",
        "month_pnl", "month_roi", "month_vlm",
        "all_time_pnl", "all_time_roi", "all_time_vlm",
    ]
    for c in numericCols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Write to Parquet (fallback to fastparquet if pyarrow has issues)
    df.to_parquet(outputPath, index=False, engine="pyarrow")
    # except Exception:
    #     df.to_parquet(outputPath, index=False, engine="fastparquet")

    print(f"Saved {len(df)} rows to {outputPath}")
    return df


fetchHyperliquidLeaderboard()

Saved 40414 rows to ../output/hyperliquidLeaderboard.parquet


,ethAddress,accountValue,prize,displayName,day_pnl,day_roi,day_vlm,week_pnl,week_roi,week_vlm,month_pnl,month_roi,month_vlm,allTime_pnl,allTime_roi,allTime_vlm
0,0x85ecf584f25db6f146718b86d493e33c5af72052,6.197768e+07,0,NaN,1.591153e+05,0.002958,1.662808e+09,-1.934849e+05,-0.003536,7.674220e+09,-3.754329e+05,-0.004189,3.488014e+10,4.118391e+06,0.037362,1.235944e+11
1,0xf5d81a135f756ca16544e53c20fc20643ec3ad53,4.796357e+07,0,NaN,-2.182750e+06,-0.043541,1.916523e+09,-3.019556e+06,-0.031691,9.404006e+09,-3.056025e+06,-0.031937,3.022403e+10,-2.378365e+06,-0.024934,9.859451e+10
2,0x87f9cd15f5050a9283b8896300f7c8cf69ece2cf,8.233011e+07,0,NaN,9.051351e+05,0.017094,1.000435e+09,1.322913e+06,0.023628,4.575211e+09,-1.613404e+06,-0.022810,2.317555e+10,7.112332e+07,0.423525,5.484652e+11
3,0x399965e15d4e61ec3529cc98b7f7ebb93b733336,1.481157e+07,0,NaN,1.604312e+04,0.001937,1.089286e+09,1.050614e+05,0.010914,6.209910e+09,8.900554e+05,0.077253,2.302343e+10,9.340135e+06,0.827216,2.006888e+11
4,0x7839e2f2c375dd2935193f2736167514efff9916,1.239913e+07,0,NaN,-1.206759e+05,-0.019328,9.350874e+08,-1.077472e+05,-0.016703,3.887248e+09,7.935640e+05,0.116513,1.891857e+10,1.687579e+07,2.105810,2.274870e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40409,0xfff0780db25c29d7b9936dd46a90273c7e8affc4,1.760590e-01,0,NaN,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,-2.067997e+06,-0.986893,1.658467e+08
40410,0xfff09ba669aaf9e472824858d1ca54a8a1ee6758,9.317135e+03,0,NaN,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,-2.694585e+05,-0.795588,4.055890e+07
40411,0xfff81c5b5882769f2a4b8a6537b7251c9bc00005,1.452793e+00,0,NaN,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,5.424200e-02,0.000542,0.000000e+00,7.628333e+04,1.779528,1.228684e+07
40412,0xfffafff1445fb2d469b7f2d87fb3eadb8b1aa87e,3.475438e+00,0,NaN,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,1.098915e+05,0.310942,2.298225e+07
